# Sentinel Pro v1.1 — Full Model Training
**Author:** @who_is_the_black_hat

## Models trained in this notebook:
1. **SentinelNet v5.0** — CNN+Transformer threat classifier (47K samples)
2. **Seq2Seq v2.0** — CNN Encoder + Transformer Decoder (27K samples)
3. **PayloadNet v1.0** — Proxy payload attack classifier (17K samples)

## Upload to Google Drive root before running:
- `sentinel_threat_v1.1.jsonl`
- `sentinel_seq2seq_v1.1.jsonl`
- `sentinel_proxy_v1.1.jsonl`

## After training, copy to Kali:
```bash
cp ~/Downloads/sentinel_threat_net.pt     /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_vocab.json        /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_seq2seq.pt        /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_seq2seq_vocab.json /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_proxy_net.pt      /home/kali/osints/models/ml_engine/
cp ~/Downloads/sentinel_proxy_vocab.json  /home/kali/osints/models/ml_engine/
```

In [ ]:
# ── Cell 1: GPU Check + Install ──────────────────────────────────────────────
import torch
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA :', torch.cuda.is_available())
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'GPU nahi mila — Runtime > Change runtime type > T4 GPU select karo'
DEVICE = torch.device('cuda')
print('Device:', DEVICE)

In [ ]:
# ── Cell 2: Mount Drive + Load All Data ──────────────────────────────────────
from google.colab import drive
from pathlib import Path
import json, glob, random
from collections import Counter, defaultdict

random.seed(42)
drive.mount('/content/drive', force_remount=True)

THREAT_LABELS = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
LABEL2IDX     = {l: i for i, l in enumerate(THREAT_LABELS)}
IDX2LABEL     = {i: l for i, l in enumerate(THREAT_LABELS)}

THREAT_TYPES  = ['recon','web_vuln','breach','malware','phishing',
                 'apt','misconfig','exploit','social_eng','unknown']
TYPE2IDX      = {t: i for i, t in enumerate(THREAT_TYPES)}
IDX2TYPE      = {i: t for i, t in enumerate(THREAT_TYPES)}

ACTION_HINTS  = ['monitor','patch_now','block_ip','escalate',
                 'investigate','notify_team','collect_evidence','no_action']
HINT2IDX      = {h: i for i, h in enumerate(ACTION_HINTS)}
IDX2HINT      = {i: h for i, h in enumerate(ACTION_HINTS)}

PROXY_ATTACKS = ['normal','sqli','xss','cmdi','lfi',
                 'path_traversal','ssrf','ssti','xxe']
PROXY2IDX     = {a: i for i, a in enumerate(PROXY_ATTACKS)}
IDX2PROXY     = {i: a for i, a in enumerate(PROXY_ATTACKS)}

def find_file(fname):
    found = glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True)
    if not found:
        found = glob.glob(f'/content/drive/MyDrive/{fname}')
    return found[0] if found else None

def load_jsonl(path, max_samples=60000):
    samples, seen = [], set()
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            if len(samples) >= max_samples: break
            try:
                d = json.loads(line.strip())
                key = str(d.get('text', d.get('input', d.get('payload',''))))[:60]
                if key in seen: continue
                seen.add(key)
                samples.append(d)
            except: pass
    return samples

# Load threat data
threat_path = find_file('sentinel_threat_v1.1.jsonl')
assert threat_path, 'sentinel_threat_v1.1.jsonl not found in Drive!'
threat_samples = load_jsonl(threat_path)
print(f'Threat samples: {len(threat_samples):,}')
print('Labels:', dict(Counter(s['label'] for s in threat_samples)))
print('Types :', dict(Counter(s['threat_type'] for s in threat_samples).most_common(5)))

# Load seq2seq data
seq_path = find_file('sentinel_seq2seq_v1.1.jsonl')
assert seq_path, 'sentinel_seq2seq_v1.1.jsonl not found in Drive!'
seq_samples = load_jsonl(seq_path)
print(f'\nSeq2Seq samples: {len(seq_samples):,}')
print('Tasks:', dict(Counter(s['task'] for s in seq_samples)))

# Load proxy data
proxy_path = find_file('sentinel_proxy_v1.1.jsonl')
assert proxy_path, 'sentinel_proxy_v1.1.jsonl not found in Drive!'
proxy_samples = load_jsonl(proxy_path)
print(f'\nProxy samples: {len(proxy_samples):,}')
print('Attacks:', dict(Counter(s['attack_type'] for s in proxy_samples)))

print('\nAll data loaded!')

In [ ]:
# ── Cell 3: Shared Tokenizer + Utilities ─────────────────────────────────────
import re, math, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class SentinelTokenizer:
    """Security-aware tokenizer — security terms ko boost karta hai vocab mein."""
    SECURITY_TERMS = {
        'ransomware','malware','exploit','backdoor','trojan','botnet',
        'phishing','zero-day','apt','c2','sqli','xss','ssrf','rce',
        'lfi','xxe','ssti','cors','jwt','oauth','infostealer','keylogger',
        'rootkit','fileless','mimikatz','cobalt-strike','metasploit',
        'cve','cvss','mitre','osint','exfiltration','persistence',
        'nmap','sqlmap','nikto','nuclei','gobuster','hydra','hashcat',
        'msfconsole','searchsploit','netcat','burpsuite','wireshark',
        'privilege-escalation','lateral-movement','command-and-control',
        'darkweb','tor','kali','pentest','redteam','ctf','payload',
        'injection','vulnerability','credential','breach','leak',
        'scanner','enumeration','reconnaissance','footprint',
    }
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self, max_vocab=15000):
        self.max_vocab = max_vocab
        self.word2idx  = {'<PAD>': 0, '<UNK>': 1}
        self.vocab_size = 2

    def _tokenize(self, text):
        return [w for w in re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', text.lower()) if len(w) >= 2]

    def build_vocab(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(self._tokenize(t))
        for term in self.SECURITY_TERMS:
            counter[term] = counter.get(term, 0) + 1000
        for word, _ in counter.most_common(self.max_vocab - 2):
            if word not in self.word2idx:
                self.word2idx[word] = len(self.word2idx)
        self.vocab_size = len(self.word2idx)
        print(f'Vocab: {self.vocab_size} tokens')

    def encode(self, text, max_len=300):
        ids = [self.word2idx.get(t, self.UNK_IDX) for t in self._tokenize(text)[:max_len]]
        return ids + [self.PAD_IDX] * (max_len - len(ids))

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({'word2idx': self.word2idx, 'max_vocab': self.max_vocab}, f)

def weighted_f1(preds, labels, n_classes):
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for p, l in zip(preds, labels):
        if p == l: tp[l] += 1
        else: fp[p] += 1; fn[l] += 1
    f1s, ws = [], []
    for c in range(n_classes):
        pr = tp[c] / (tp[c] + fp[c] + 1e-8)
        rc = tp[c] / (tp[c] + fn[c] + 1e-8)
        f1s.append(2 * pr * rc / (pr + rc + 1e-8))
        ws.append(labels.count(c))
    return sum(f * w for f, w in zip(f1s, ws)) / (sum(ws) or 1)

def stratified_split(samples, label_key, val_ratio=0.15):
    cls_idx = defaultdict(list)
    for i, s in enumerate(samples):
        cls_idx[s[label_key]].append(i)
    train_idx, val_idx = [], []
    for idxs in cls_idx.values():
        n = max(1, int(len(idxs) * val_ratio))
        val_idx.extend(idxs[:n])
        train_idx.extend(idxs[n:])
    return [samples[i] for i in train_idx], [samples[i] for i in val_idx]

print('Utilities ready!')

In [ ]:
# ── Cell 4: SentinelNet v5.0 Architecture ───────────────────────────────────
class SentinelNet(nn.Module):
    VERSION = "5.0"
    AUTHOR  = "who_is_the_black_hat"

    def __init__(self, vocab_size, embed_dim=128, num_filters=128,
                 kernels=(3,5,7), dropout=0.35, pad_idx=0):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_drop   = nn.Dropout(dropout)
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(embed_dim, num_filters, k, padding=k//2), nn.GELU())
            for k in kernels
        ])
        cnn_out = num_filters * len(kernels)
        self.cnn_proj   = nn.Linear(cnn_out, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head_label  = self._head(embed_dim, len(THREAT_LABELS),  dropout)
        self.head_type   = self._head(embed_dim, len(THREAT_TYPES),   dropout)
        self.head_action = self._head(embed_dim, len(ACTION_HINTS),   dropout)
        for n, p in self.named_parameters():
            if "weight" in n and p.dim() >= 2: nn.init.xavier_uniform_(p)
            elif "bias" in n: nn.init.zeros_(p)

    @staticmethod
    def _head(in_d, out_d, dr):
        return nn.Sequential(
            nn.Dropout(dr), nn.Linear(in_d, in_d//2),
            nn.GELU(), nn.Linear(in_d//2, out_d)
        )

    def forward(self, x):
        emb    = self.pos_drop(self.embedding(x)).transpose(1, 2)
        pooled = torch.cat([c(emb).max(dim=2).values for c in self.convs], dim=-1)
        ctx    = self.layer_norm(torch.relu(self.cnn_proj(pooled)))
        return self.head_label(ctx), self.head_type(ctx), self.head_action(ctx)

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {"params": p, "size_mb": round(p*4/1024/1024, 2), "version": self.VERSION}


class ThreatDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=300):
        self.encodings    = [tokenizer.encode(s["text"], max_len) for s in samples]
        self.labels       = [LABEL2IDX[s["label"]] for s in samples]
        self.threat_types = [TYPE2IDX.get(s.get("threat_type","unknown"), 9) for s in samples]
        self.action_hints = [HINT2IDX.get(s.get("action_hint","monitor"), 0) for s in samples]

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encodings[idx],    dtype=torch.long),
            torch.tensor(self.labels[idx],       dtype=torch.long),
            torch.tensor(self.threat_types[idx], dtype=torch.long),
            torch.tensor(self.action_hints[idx], dtype=torch.long),
        )


print(f"SentinelNet v5.0 defined | Device: {DEVICE}")


In [ ]:
# ── Cell 5: SentinelNet v5.0 Training ───────────────────────────────────────
MAX_LEN  = 300
BATCH    = 128
EPOCHS   = 40
PATIENCE = 8

tr_s, vl_s = stratified_split(threat_samples, "label", val_ratio=0.15)
print(f"Train: {len(tr_s):,} | Val: {len(vl_s):,}")

tokenizer = SentinelTokenizer(max_vocab=15000)
tokenizer.build_vocab([s["text"] for s in tr_s])

PIN = True
NW  = 2
tr_dl = DataLoader(ThreatDataset(tr_s, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=True,  num_workers=NW, pin_memory=PIN)
vl_dl = DataLoader(ThreatDataset(vl_s, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=False, num_workers=NW, pin_memory=PIN)

model = SentinelNet(tokenizer.vocab_size).to(DEVICE)
info  = model.info()
print(f"Params: {info['params']:,} | Size: {info['size_mb']} MB")

# Class weights — imbalanced labels handle karo
dist = Counter(LABEL2IDX[s["label"]] for s in tr_s)
w = torch.tensor([len(tr_s)/(4*dist.get(i,1)) for i in range(4)], dtype=torch.float).to(DEVICE)

crit_label  = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
crit_type   = nn.CrossEntropyLoss(label_smoothing=0.05)
crit_action = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=3e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, steps_per_epoch=len(tr_dl),
    epochs=EPOCHS, pct_start=0.2)

best_f1, best_state, no_imp = 0.0, None, 0
print(f"Training on {DEVICE}...")
print()

for epoch in range(1, EPOCHS+1):
    model.train()
    tl = 0
    for xb, yb_l, yb_t, yb_a in tr_dl:
        xb, yb_l, yb_t, yb_a = xb.to(DEVICE), yb_l.to(DEVICE), yb_t.to(DEVICE), yb_a.to(DEVICE)
        optimizer.zero_grad()
        l_l, l_t, l_a = model(xb)
        loss = 0.6*crit_label(l_l,yb_l) + 0.2*crit_type(l_t,yb_t) + 0.2*crit_action(l_a,yb_a)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        tl += loss.item()
    tl /= len(tr_dl)

    model.eval(); vl = 0; preds = []; lbls = []
    with torch.no_grad():
        for xb, yb_l, yb_t, yb_a in vl_dl:
            xb, yb_l, yb_t, yb_a = xb.to(DEVICE), yb_l.to(DEVICE), yb_t.to(DEVICE), yb_a.to(DEVICE)
            l_l, l_t, l_a = model(xb)
            vl += (0.6*crit_label(l_l,yb_l) + 0.2*crit_type(l_t,yb_t) + 0.2*crit_action(l_a,yb_a)).item()
            preds.extend(l_l.argmax(1).cpu().tolist())
            lbls.extend(yb_l.cpu().tolist())
    vl /= len(vl_dl)
    acc = sum(p==l for p,l in zip(preds,lbls)) / len(lbls)
    f1  = weighted_f1(preds, lbls, 4)
    print(f"Epoch {epoch:2d} | train={tl:.4f} | val={vl:.4f} | acc={acc:.2%} | f1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_imp = 0
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"Early stop @ epoch {epoch}")
            break

model.load_state_dict(best_state)
print("\nBest F1:", round(best_f1, 4))


In [ ]:
# ── Cell 6: SentinelNet Save + Test ─────────────────────────────────────────
from google.colab import files

checkpoint = {
    "model_state":  model.state_dict(),
    "model_config": model.info(),
    "hyperparams": {
        "embed_dim":   128,
        "num_filters": 128,
        "tf_layers":   2,
        "dropout":     0.35,
        "max_len":     MAX_LEN,
    },
    "version":       "5.0",
    "author":        "who_is_the_black_hat",
    "saved_at":      time.strftime("%Y-%m-%d %H:%M:%S"),
    "labels":        THREAT_LABELS,
    "threat_types":  THREAT_TYPES,
    "action_hints":  ACTION_HINTS,
    "best_f1":       best_f1,
    "train_samples": len(tr_s),
}
torch.save(checkpoint, "sentinel_threat_net.pt")
tokenizer.save("sentinel_vocab.json")

print(f"Best F1  : {best_f1:.4f}")
print(f"Params   : {model.info()['params']:,}")
print(f"Size     : {model.info()['size_mb']} MB")

# Quick inference test
def predict_threat(text):
    model.eval()
    ids = tokenizer.encode(text, MAX_LEN)
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        l_l, l_t, l_a = model(x)
        p_l = F.softmax(l_l, dim=-1)[0]
        p_t = F.softmax(l_t, dim=-1)[0]
        p_a = F.softmax(l_a, dim=-1)[0]
    return {
        "label":       IDX2LABEL[p_l.argmax().item()],
        "threat_type": IDX2TYPE[p_t.argmax().item()],
        "action_hint": IDX2HINT[p_a.argmax().item()],
        "confidence":  round(p_l.max().item(), 4),
    }

tests = [
    "sqlmap found sql injection vulnerability database dumped credentials exposed",
    "ransomware encrypted all files bitcoin ransom demand critical infrastructure",
    "nmap scan found open ports 22 80 443 running services detected",
    "security headers missing x-content-type csp hsts not configured",
    "normal website no vulnerabilities found ssl grade a headers ok",
]
print("\n=== Inference Test ===")
for t in tests:
    r = predict_threat(t)
    print(f"[{r['label']:8s}] type={r['threat_type']:12s} action={r['action_hint']:18s} conf={r['confidence']:.2f}")
    print(f"  > {t[:70]}")
    print()

files.download("sentinel_threat_net.pt")
files.download("sentinel_vocab.json")
print("Downloaded!")


In [ ]:
# ── Cell 7: Seq2Seq Tokenizer + Architecture ─────────────────────────────────
class SeqTokenizer:
    PAD, BOS, EOS, UNK = 0, 1, 2, 3
    SPECIAL = ["<PAD>","<BOS>","<EOS>","<UNK>"]
    SEC_TERMS = {
        "nmap","sqlmap","nikto","nuclei","gobuster","ffuf","amass","subfinder",
        "whatweb","wafw00f","sslscan","theharvester","searchsploit","commix",
        "wpscan","enum4linux","hydra","masscan","hashcat","john","target",
        "scan","exploit","vulnerability","injection","xss","sqli","rce",
        "lfi","xxe","ssti","cors","jwt","breach","malware","phishing","apt",
        "recon","misconfig","critical","high","medium","low","patch","escalate",
    }

    def __init__(self, max_vocab=10000):
        self.max_vocab = max_vocab
        self.w2i = {s: i for i, s in enumerate(self.SPECIAL)}
        self.i2w = {i: s for i, s in enumerate(self.SPECIAL)}
        self.vocab_size = len(self.SPECIAL)

    def _tok(self, text):
        return [w for w in re.findall(r"[a-z0-9]+(?:[._/-][a-z0-9]+)*", text.lower()) if len(w) >= 1]

    def build_vocab(self, texts):
        cnt = Counter()
        for t in texts: cnt.update(self._tok(t))
        for term in self.SEC_TERMS: cnt[term] = cnt.get(term, 0) + 5000
        for w, _ in cnt.most_common(self.max_vocab - len(self.SPECIAL)):
            if w not in self.w2i:
                idx = len(self.w2i)
                self.w2i[w] = idx
                self.i2w[idx] = w
        self.vocab_size = len(self.w2i)
        print(f"Seq vocab: {self.vocab_size}")

    def encode_src(self, text, max_len=128):
        ids = [self.w2i.get(w, self.UNK) for w in self._tok(text)[:max_len]]
        return ids + [self.PAD] * (max_len - len(ids))

    def encode_tgt(self, text, max_len=64):
        ids = [self.BOS] + [self.w2i.get(w, self.UNK) for w in self._tok(text)[:max_len-2]] + [self.EOS]
        return ids + [self.PAD] * (max_len - len(ids))

    def decode(self, ids):
        words = []
        for i in ids:
            if i == self.EOS: break
            if i in (self.PAD, self.BOS): continue
            w = self.i2w.get(i, "")
            if w: words.append(w)
        return " ".join(words)

    def save(self, path):
        json.dump({"w2i": self.w2i, "i2w": {str(k):v for k,v in self.i2w.items()},
                   "max_vocab": self.max_vocab}, open(path, "w"))


class Seq2SeqDataset(Dataset):
    def __init__(self, samples, tok, src_len=128, tgt_len=64):
        self.src = [tok.encode_src(s["input"],  src_len) for s in samples]
        self.tgt = [tok.encode_tgt(s["output"], tgt_len) for s in samples]

    def __len__(self): return len(self.src)

    def __getitem__(self, i):
        return (torch.tensor(self.src[i], dtype=torch.long),
                torch.tensor(self.tgt[i], dtype=torch.long))


class SentinelSeq2Seq(nn.Module):
    VERSION = "2.0"
    AUTHOR  = "who_is_the_black_hat"

    def __init__(self, vocab_size, embed_dim=256, num_filters=256,
                 nhead=4, dec_layers=3, ff_dim=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx   = pad_idx
        self.emb_scale = math.sqrt(embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_drop  = nn.Dropout(dropout)
        self.enc_convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(embed_dim, num_filters, k, padding=k//2),
                          nn.GELU(), nn.Dropout(dropout))
            for k in (3, 5, 7)
        ])
        self.enc_proj = nn.Linear(num_filters * 3, embed_dim)
        self.enc_norm = nn.LayerNorm(embed_dim)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True)
        self.decoder  = nn.TransformerDecoder(dec_layer, num_layers=dec_layers)
        self.out_proj = nn.Linear(embed_dim, vocab_size)
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def _pos_enc(self, x):
        B, L, D = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, D, 2, device=x.device).float() * (-math.log(10000.0) / D))
        pe  = torch.zeros(L, D, device=x.device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:D//2])
        return x + pe.unsqueeze(0)

    def encode(self, src):
        emb   = self.emb_drop(self.embedding(src) * self.emb_scale)
        x     = emb.transpose(1, 2)
        feats = torch.cat([c(x).transpose(1,2) for c in self.enc_convs], dim=-1)
        return self.enc_norm(F.gelu(self.enc_proj(feats)))

    def forward(self, src, tgt):
        enc_out = self.encode(src)
        tgt_in  = tgt[:, :-1]
        tgt_emb = self._pos_enc(self.emb_drop(self.embedding(tgt_in) * self.emb_scale))
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
        src_pad = (src == self.pad_idx)
        tgt_pad = (tgt_in == self.pad_idx)
        out = self.decoder(tgt_emb, enc_out, tgt_mask=causal,
                           tgt_key_padding_mask=tgt_pad,
                           memory_key_padding_mask=src_pad)
        return self.out_proj(out)

    def greedy(self, src, tok, max_len=64):
        self.eval()
        with torch.no_grad():
            enc_out = self.encode(src)
            gen = [tok.BOS]
            for _ in range(max_len):
                tgt_ids = torch.tensor([gen], dtype=torch.long, device=src.device)
                tgt_emb = self._pos_enc(self.emb_drop(self.embedding(tgt_ids) * self.emb_scale))
                T = tgt_ids.size(1)
                causal = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
                out = self.decoder(tgt_emb, enc_out, tgt_mask=causal)
                nxt = self.out_proj(out[:, -1, :]).argmax(-1).item()
                if nxt == tok.EOS: break
                gen.append(nxt)
        return tok.decode(gen[1:])

    def generate(self, src, tok, max_len=64, temperature=0.7, top_k=50):
        self.eval()
        with torch.no_grad():
            enc_out = self.encode(src)
            gen = [tok.BOS]
            for _ in range(max_len):
                tgt_ids = torch.tensor([gen], dtype=torch.long, device=src.device)
                tgt_emb = self._pos_enc(self.emb_drop(self.embedding(tgt_ids) * self.emb_scale))
                T = tgt_ids.size(1)
                causal = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
                out    = self.decoder(tgt_emb, enc_out, tgt_mask=causal)
                logits = self.out_proj(out[:, -1, :]) / temperature
                if top_k > 0:
                    vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < vals[:, -1:]] = float("-inf")
                probs = F.softmax(logits, dim=-1)
                nxt   = torch.multinomial(probs, 1).item()
                if nxt == tok.EOS: break
                gen.append(nxt)
        return tok.decode(gen[1:])

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {"params": p, "size_mb": round(p*4/1024/1024, 2), "version": self.VERSION}


print(f"SentinelSeq2Seq v2.0 defined | Device: {DEVICE}")


In [ ]:
# ── Cell 8: Seq2Seq Training (3-Round Curriculum) ────────────────────────────
SRC_LEN = 128
TGT_LEN = 64

# Build vocab from all seq samples
seq_tok = SeqTokenizer(max_vocab=10000)
seq_tok.build_vocab([s["input"] for s in seq_samples] + [s["output"] for s in seq_samples])

# Split by task
task_buckets = defaultdict(list)
for s in seq_samples:
    task_buckets[s["task"]].append(s)

print("Task distribution:")
for task, items in task_buckets.items():
    print(f"  {task}: {len(items):,}")

# Curriculum: report_gen (largest) -> chain_gen -> cmd_gen
# Combine all for round 1, then fine-tune on smaller tasks
r1 = task_buckets.get("report_gen", [])   # Round 1: report_gen
r2 = task_buckets.get("chain_gen", [])    # Round 2: chain_gen
r3 = task_buckets.get("cmd_gen", [])      # Round 3: cmd_gen

# Safety check — agar koi bucket empty hai toh skip karo
assert len(r1) > 0, "report_gen samples missing!"
assert len(r2) > 0, "chain_gen samples missing!"
print(f"R1 report_gen: {len(r1):,} | R2 chain_gen: {len(r2):,} | R3 cmd_gen: {len(r3):,}")

def make_loaders(samples, batch_size=128):
    random.shuffle(samples)
    n_val = max(50, int(len(samples) * 0.1))
    tr, vl = samples[n_val:], samples[:n_val]
    tr_dl = DataLoader(Seq2SeqDataset(tr, seq_tok, SRC_LEN, TGT_LEN),
                       batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    vl_dl = DataLoader(Seq2SeqDataset(vl, seq_tok, SRC_LEN, TGT_LEN),
                       batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return tr_dl, vl_dl

def train_round(seq_model, samples, round_num, lr, epochs, patience):
    if len(samples) < 10:
        print(f"Round {round_num}: skipped (only {len(samples)} samples)")
        return 0.0
    print("\n" + "="*55)
    print(f"ROUND {round_num} | {len(samples):,} samples | LR={lr} | Epochs={epochs}")
    print("="*55)
    tr_dl, vl_dl = make_loaders(samples)
    criterion = nn.CrossEntropyLoss(ignore_index=seq_tok.PAD, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(seq_model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(tr_dl),
        epochs=epochs, pct_start=0.1, anneal_strategy="cos")
    best_loss, best_state, no_imp = float("inf"), None, 0
    for epoch in range(1, epochs+1):
        seq_model.train()
        tl = 0
        for src, tgt in tr_dl:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            optimizer.zero_grad()
            logits = seq_model(src, tgt)
            B, T1, V = logits.shape
            loss = criterion(logits.reshape(B*T1, V), tgt[:, 1:].reshape(B*T1))
            loss.backward()
            nn.utils.clip_grad_norm_(seq_model.parameters(), 0.5)
            optimizer.step(); scheduler.step()
            tl += loss.item()
        tl /= len(tr_dl)
        seq_model.eval(); vl = 0; acc = 0
        with torch.no_grad():
            for src, tgt in vl_dl:
                src, tgt = src.to(DEVICE), tgt.to(DEVICE)
                logits = seq_model(src, tgt)
                B, T1, V = logits.shape
                vl += criterion(logits.reshape(B*T1, V), tgt[:, 1:].reshape(B*T1)).item()
                pred = logits.argmax(-1); gold = tgt[:, 1:]
                mask = gold != seq_tok.PAD
                acc += (pred[mask] == gold[mask]).float().mean().item()
        vl /= len(vl_dl); acc /= len(vl_dl)
        print(f"  Epoch {epoch:2d} | train={tl:.4f} | val={vl:.4f} | acc={acc:.2%}")
        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.clone() for k, v in seq_model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"  Early stop @ epoch {epoch}")
                break
    seq_model.load_state_dict(best_state)
    print(f"  Best val loss: {best_loss:.4f}")
    return best_loss

# Init model
seq_model = SentinelSeq2Seq(vocab_size=seq_tok.vocab_size).to(DEVICE)
print(f"Seq2Seq params: {seq_model.info()['params']:,} | Size: {seq_model.info()['size_mb']} MB")

# 3-Round curriculum
loss_r1 = train_round(seq_model, r1, round_num=1, lr=3e-4, epochs=15, patience=4)
loss_r2 = train_round(seq_model, r2, round_num=2, lr=1e-4, epochs=15, patience=4)
loss_r3 = train_round(seq_model, r3, round_num=3, lr=5e-5, epochs=15, patience=4)

print(f"Curriculum complete | R1={loss_r1:.4f} R2={loss_r2:.4f} R3={loss_r3:.4f}")


In [ ]:
# ── Cell 9: Seq2Seq Save + Test ──────────────────────────────────────────────
checkpoint_seq = {
    "model_state":  seq_model.state_dict(),
    "model_config": seq_model.info(),
    "hyperparams": {
        "vocab_size":  seq_tok.vocab_size,
        "embed_dim":   256,
        "num_filters": 256,
        "nhead":       4,
        "dec_layers":  3,
        "ff_dim":      512,
        "dropout":     0.1,
        "src_len":     SRC_LEN,
        "tgt_len":     TGT_LEN,
    },
    "version":   "2.0",
    "author":    "who_is_the_black_hat",
    "saved_at":  time.strftime("%Y-%m-%d %H:%M:%S"),
    "tasks":     ["cmd_gen", "chain_gen", "report_gen"],
    "training":  {"r1_loss": loss_r1, "r2_loss": loss_r2, "r3_loss": loss_r3},
}
torch.save(checkpoint_seq, "sentinel_seq2seq.pt")
seq_tok.save("sentinel_seq2seq_vocab.json")

print(f"Size: {seq_model.info()['size_mb']} MB | Params: {seq_model.info()['params']:,}")

# Inference test
def encode_src(text):
    ids = seq_tok.encode_src(text, SRC_LEN)
    return torch.tensor([ids], dtype=torch.long).to(DEVICE)

tests = [
    ("[SCAN_CONTEXT] web server port 80 open need vulnerability scan [THREAT] HIGH [TYPE] web_vuln", "cmd_gen"),
    ("[SCAN_CONTEXT] sql injection suspected login form [THREAT] CRITICAL [TYPE] web_vuln", "cmd_gen"),
    ("[CURRENT_TOOL] nmap [FINDING] open web port 80 443 [STATE] scan in progress [THREAT] HIGH [TYPE] recon", "chain_gen"),
    ("[CURRENT_TOOL] nikto [FINDING] xss found admin panel [STATE] scan in progress [THREAT] HIGH [TYPE] web_vuln", "chain_gen"),
    ("[FINDINGS] ransomware encrypted files [SEVERITY] CRITICAL [TYPE] malware [ACTION] escalate", "report_gen"),
    ("[FINDINGS] sql injection found login form [SEVERITY] CRITICAL [TYPE] web_vuln [ACTION] patch_now", "report_gen"),
]
print("\n=== Seq2Seq Inference Test ===")
for inp, task in tests:
    src = encode_src(inp)
    if task == "chain_gen":
        out = seq_model.greedy(src, seq_tok)
    else:
        out = seq_model.generate(src, seq_tok, temperature=0.7, top_k=50)
    print(f"[{task}] {inp[:55]}")
    print(f"  -> {out}")
    print()

files.download("sentinel_seq2seq.pt")
files.download("sentinel_seq2seq_vocab.json")
print("Downloaded!")


In [ ]:
# ── Cell 10: PayloadNet v1.0 — Proxy Payload Classifier ──────────────────────
# Architecture: Char-level CNN + Multi-class classifier
# Input: raw HTTP payload string (char-level tokenization)
# Output: attack_type (normal/sqli/xss/cmdi/lfi/path_traversal/ssrf/ssti/xxe)

PROXY_ATTACKS = ["normal","sqli","xss","cmdi","lfi","path_traversal","ssrf","ssti","xxe"]
PROXY2IDX     = {a: i for i, a in enumerate(PROXY_ATTACKS)}
IDX2PROXY     = {i: a for i, a in enumerate(PROXY_ATTACKS)}

# Char-level tokenizer for payloads
class PayloadTokenizer:
    CHARS = list("abcdefghijklmnopqrstuvwxyz0123456789 .,;:!?/-_=+*&|<>()[]{}@#$%^~`")
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self):
        self.char2idx = {"<PAD>": 0, "<UNK>": 1}
        for c in self.CHARS:
            self.char2idx[c] = len(self.char2idx)
        self.vocab_size = len(self.char2idx)

    def encode(self, text, max_len=200):
        ids = [self.char2idx.get(c.lower(), self.UNK_IDX) for c in text[:max_len]]
        return ids + [self.PAD_IDX] * (max_len - len(ids))

    def save(self, path):
        json.dump({"char2idx": self.char2idx, "vocab_size": self.vocab_size}, open(path, "w"))


class PayloadDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=200):
        self.encodings = [tokenizer.encode(s["payload"], max_len) for s in samples]
        self.labels    = [PROXY2IDX[s["attack_type"]] for s in samples]

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (torch.tensor(self.encodings[idx], dtype=torch.long),
                torch.tensor(self.labels[idx],    dtype=torch.long))


class PayloadNet(nn.Module):
    """Char-level CNN classifier for HTTP payload attack detection."""
    VERSION = "1.0"
    AUTHOR  = "who_is_the_black_hat"

    def __init__(self, vocab_size, embed_dim=64, num_filters=128,
                 kernels=(3,5,7,11), dropout=0.4, pad_idx=0):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_drop   = nn.Dropout(dropout)
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embed_dim, num_filters, k, padding=k//2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
            ) for k in kernels
        ])
        cnn_out = num_filters * len(kernels)
        self.proj      = nn.Linear(cnn_out, 256)
        self.norm      = nn.LayerNorm(256)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, len(PROXY_ATTACKS)),
        )
        for n, p in self.named_parameters():
            if "weight" in n and p.dim() >= 2: nn.init.xavier_uniform_(p)
            elif "bias" in n: nn.init.zeros_(p)

    def forward(self, x):
        emb    = self.emb_drop(self.embedding(x)).transpose(1, 2)
        pooled = torch.cat([c(emb).max(dim=2).values for c in self.convs], dim=-1)
        ctx    = self.norm(F.gelu(self.proj(pooled)))
        return self.classifier(ctx)

    def info(self):
        p = sum(x.numel() for x in self.parameters())
        return {"params": p, "size_mb": round(p*4/1024/1024, 2), "version": self.VERSION}


# Train PayloadNet
PROXY_MAX_LEN = 200
PROXY_BATCH   = 256
PROXY_EPOCHS  = 30
PROXY_PATIENCE = 6

proxy_tok = PayloadTokenizer()
print(f"Payload vocab: {proxy_tok.vocab_size} chars")

pr_tr, pr_vl = stratified_split(proxy_samples, "attack_type", val_ratio=0.15)
print(f"Proxy Train: {len(pr_tr):,} | Val: {len(pr_vl):,}")
print("Train dist:", dict(Counter(s["attack_type"] for s in pr_tr)))

pr_tr_dl = DataLoader(PayloadDataset(pr_tr, proxy_tok, PROXY_MAX_LEN),
                      batch_size=PROXY_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
pr_vl_dl = DataLoader(PayloadDataset(pr_vl, proxy_tok, PROXY_MAX_LEN),
                      batch_size=PROXY_BATCH, shuffle=False, num_workers=2, pin_memory=True)

proxy_model = PayloadNet(proxy_tok.vocab_size).to(DEVICE)
print(f"PayloadNet params: {proxy_model.info()['params']:,} | Size: {proxy_model.info()['size_mb']} MB")

# Class weights
proxy_dist = Counter(PROXY2IDX[s["attack_type"]] for s in pr_tr)
proxy_w = torch.tensor([len(pr_tr)/(len(PROXY_ATTACKS)*proxy_dist.get(i,1))
                         for i in range(len(PROXY_ATTACKS))], dtype=torch.float).to(DEVICE)
proxy_crit = nn.CrossEntropyLoss(weight=proxy_w, label_smoothing=0.05)

proxy_opt  = torch.optim.AdamW(proxy_model.parameters(), lr=5e-4, weight_decay=1e-4)
proxy_sched = torch.optim.lr_scheduler.OneCycleLR(
    proxy_opt, max_lr=5e-4, steps_per_epoch=len(pr_tr_dl),
    epochs=PROXY_EPOCHS, pct_start=0.2)

best_proxy_f1, best_proxy_state, no_imp = 0.0, None, 0
print("\nTraining PayloadNet on", DEVICE, "...")
print()

for epoch in range(1, PROXY_EPOCHS+1):
    proxy_model.train()
    tl = 0
    for xb, yb in pr_tr_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        proxy_opt.zero_grad()
        loss = proxy_crit(proxy_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(proxy_model.parameters(), 1.0)
        proxy_opt.step(); proxy_sched.step()
        tl += loss.item()
    tl /= len(pr_tr_dl)

    proxy_model.eval(); preds = []; lbls = []
    with torch.no_grad():
        for xb, yb in pr_vl_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds.extend(proxy_model(xb).argmax(1).cpu().tolist())
            lbls.extend(yb.cpu().tolist())
    acc = sum(p==l for p,l in zip(preds,lbls)) / len(lbls)
    f1  = weighted_f1(preds, lbls, len(PROXY_ATTACKS))
    print(f"Epoch {epoch:2d} | train={tl:.4f} | acc={acc:.2%} | f1={f1:.4f}")

    if f1 > best_proxy_f1:
        best_proxy_f1 = f1
        best_proxy_state = {k: v.clone() for k, v in proxy_model.state_dict().items()}
        no_imp = 0
    else:
        no_imp += 1
        if no_imp >= PROXY_PATIENCE:
            print(f"Early stop @ epoch {epoch}")
            break

proxy_model.load_state_dict(best_proxy_state)
print(f"\nBest PayloadNet F1: {best_proxy_f1:.4f}")


In [ ]:
# ── Cell 11: PayloadNet Save + Test + Final Download ─────────────────────────
checkpoint_proxy = {
    "model_state":  proxy_model.state_dict(),
    "model_config": proxy_model.info(),
    "hyperparams": {
        "vocab_size":  proxy_tok.vocab_size,
        "embed_dim":   64,
        "num_filters": 128,
        "kernels":     [3, 5, 7, 11],
        "dropout":     0.4,
        "max_len":     PROXY_MAX_LEN,
    },
    "version":      "1.0",
    "author":       "who_is_the_black_hat",
    "saved_at":     time.strftime("%Y-%m-%d %H:%M:%S"),
    "attack_types": PROXY_ATTACKS,
    "best_f1":      best_proxy_f1,
    "train_samples": len(pr_tr),
}
torch.save(checkpoint_proxy, "sentinel_proxy_net.pt")
proxy_tok.save("sentinel_proxy_vocab.json")

# Inference test
def predict_payload(payload):
    proxy_model.eval()
    ids = proxy_tok.encode(payload, PROXY_MAX_LEN)
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = proxy_model(x)
        probs  = F.softmax(logits, dim=-1)[0]
    return {
        "attack_type": IDX2PROXY[probs.argmax().item()],
        "confidence":  round(probs.max().item(), 4),
        "is_malicious": IDX2PROXY[probs.argmax().item()] != "normal",
    }

test_payloads = [
    ("normal user input",                              "normal"),
    ("john.doe@example.com",                          "normal"),
    ("1 OR 1=1--",                                    "sqli"),
    ("UNION SELECT NULL,NULL,NULL--",                  "sqli"),
    ("<script>alert(1)</script>",                     "xss"),
    ("<img src=x onerror=alert(1)>",                  "xss"),
    ("../../etc/passwd",                              "lfi"),
    ("; cat /etc/passwd",                             "cmdi"),
    ("http://169.254.169.254/latest/meta-data/",      "ssrf"),
    ("{{7*7}}",                                       "ssti"),
    ("../../../windows/win.ini",                      "path_traversal"),
]
print("\n=== PayloadNet Inference Test ===")
correct = 0
for payload, expected in test_payloads:
    r = predict_payload(payload)
    ok = "OK" if r["attack_type"] == expected else "WRONG"
    if ok == "OK": correct += 1
    print(f"[{ok}] expected={expected:15s} got={r['attack_type']:15s} conf={r['confidence']:.2f}")
    print(f"     payload: {payload[:60]}")
print(f"\nAccuracy: {correct}/{len(test_payloads)} = {correct/len(test_payloads):.0%}")

# ── Final Summary ─────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("TRAINING COMPLETE — FINAL SUMMARY")
print("="*55)
print(f"SentinelNet v5.0  | F1={best_f1:.4f}       | {model.info()['size_mb']} MB")
print(f"Seq2Seq v2.0      | R3_loss={loss_r3:.4f}  | {seq_model.info()['size_mb']} MB")
print(f"PayloadNet v1.0   | F1={best_proxy_f1:.4f} | {proxy_model.info()['size_mb']} MB")
print("="*55)

# Download all files
print("\nDownloading all model files...")
files.download("sentinel_threat_net.pt")
files.download("sentinel_vocab.json")
files.download("sentinel_seq2seq.pt")
files.download("sentinel_seq2seq_vocab.json")
files.download("sentinel_proxy_net.pt")
files.download("sentinel_proxy_vocab.json")
print("All downloaded!")

print("\nKali pe copy karo:")
print("cp ~/Downloads/sentinel_threat_net.pt      /home/kali/osints/models/ml_engine/")
print("cp ~/Downloads/sentinel_vocab.json         /home/kali/osints/models/ml_engine/")
print("cp ~/Downloads/sentinel_seq2seq.pt         /home/kali/osints/models/ml_engine/")
print("cp ~/Downloads/sentinel_seq2seq_vocab.json /home/kali/osints/models/ml_engine/")
print("cp ~/Downloads/sentinel_proxy_net.pt       /home/kali/osints/models/ml_engine/")
print("cp ~/Downloads/sentinel_proxy_vocab.json   /home/kali/osints/models/ml_engine/")
